# 33 - Analyze source action horizon

This notebook compares the source model with **10 executed actions per generated chunk** for the two arms collected by notebooks 31 and 32:

- `pnp_uncertainty_only`: no refinement
- `pnp_refinement`: k=5, Euler steps (3, 4), last-sample refinement

Rows from the earlier 20-action run share the same experiment name, so this notebook explicitly filters `config_json.n_action_steps == 10` before matching.

## 1. Setup

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Fetch only the 10-action rows and exact-match the two arms

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from analysis.statistics import paired_bootstrap_ci, discordant_test
from pnp.config import Method
from pnp.diversity import DIVERSITY_PAIR_KEYS, SOURCE_ACTION_HORIZON_EXPERIMENT
from pnp.store import SupabaseStore

N_ACTION_STEPS = 10
EXPECTED_FULL_COHORT = 1300
REQUIRE_FULL_MATCHED_COHORT = False
OUTPUT = Path('source_action_horizon_10_outputs')
OUTPUT.mkdir(exist_ok=True)
store = SupabaseStore()

rows = pd.DataFrame(store.fetch_all(
    'rollouts', '*', configure=lambda query: query.eq(
        'experiment', SOURCE_ACTION_HORIZON_EXPERIMENT).in_(
            'method', [Method.UNCERTAINTY, Method.REFINEMENT]),
    order_by=('rollout_id',)))
assert len(rows), f'No rows found for {SOURCE_ACTION_HORIZON_EXPERIMENT}'

def logical_config(value):
    if isinstance(value, str):
        value = json.loads(value)
    return value or {}

def step_indices(value):
    if isinstance(value, str):
        value = json.loads(value)
    return tuple(value or [])

rows = rows[rows.status.eq('completed')].copy()
rows['logical_config'] = rows.config_json.apply(logical_config)
rows['n_action_steps_logged'] = rows.logical_config.apply(
    lambda value: value.get('n_action_steps'))
print('Completed rows by logged execution horizon and method:')
display(rows.groupby(['n_action_steps_logged', 'method'], dropna=False).size()
        .rename('rows').reset_index())

selected = rows[
    rows.n_action_steps_logged.eq(N_ACTION_STEPS)
    & rows.pnp_k.eq(5)
    & rows.pnp_step_indices.apply(lambda value: step_indices(value) == (3, 4))
].copy()
assert len(selected), f'No completed rows found with n_action_steps={N_ACTION_STEPS}'

baseline = selected[selected.method.eq(Method.UNCERTAINTY)].copy()
refined = selected[selected.method.eq(Method.REFINEMENT)].copy()
for name, frame in (('baseline', baseline), ('refinement', refined)):
    assert len(frame), f'No {name} rows remain after filtering'
    assert not frame.duplicated(DIVERSITY_PAIR_KEYS).any(), (
        f'Duplicate {name} episode identities after filtering')

keep = DIVERSITY_PAIR_KEYS + ['rollout_id', 'success', 'n_steps', 'n_chunks',
                              'config_hash']
paired = (baseline[keep].rename(columns={
              'rollout_id': 'baseline_rollout_id',
              'success': 'baseline_success',
              'n_steps': 'baseline_n_steps',
              'n_chunks': 'baseline_n_chunks',
              'config_hash': 'baseline_config_hash'})
          .merge(refined[keep].rename(columns={
              'rollout_id': 'refinement_rollout_id',
              'success': 'refinement_success',
              'n_steps': 'refinement_n_steps',
              'n_chunks': 'refinement_n_chunks',
              'config_hash': 'refinement_config_hash'}),
                 on=DIVERSITY_PAIR_KEYS, validate='one_to_one'))
for column in ('baseline_success', 'refinement_success'):
    paired[column] = paired[column].astype(bool)

coverage = pd.DataFrame([{
    '10_action_baseline_rows': len(baseline),
    '10_action_refinement_rows': len(refined),
    'exact_matched_episodes': len(paired),
    'matched_suites': paired.suite.nunique(),
    'full_1300_episode_coverage_pct': 100 * len(paired) / EXPECTED_FULL_COHORT,
}])
display(coverage)
assert len(paired), 'The two 10-action arms have no exact-matched episodes'
assert len(paired) == len(refined), (
    f'Only {len(paired)}/{len(refined)} refinement rows match the 10-action baseline')
if REQUIRE_FULL_MATCHED_COHORT:
    assert len(paired) == EXPECTED_FULL_COHORT, (
        f'Expected {EXPECTED_FULL_COHORT} matched episodes, found {len(paired)}')
print('Only exact matches on suite/task/episode/init-state enter every SR below.')

## 3. Overall and per-suite paired result

In [ ]:
def summarize(group):
    baseline_values = group.baseline_success.to_numpy(bool)
    refinement_values = group.refinement_success.to_numpy(bool)
    lo, hi = paired_bootstrap_ci(baseline_values, refinement_values, n_boot=5000)
    f_to_s = int((~baseline_values & refinement_values).sum())
    s_to_f = int((baseline_values & ~refinement_values).sum())
    return pd.Series({
        'episodes': len(group),
        '10_action_baseline_sr_pct': 100 * baseline_values.mean(),
        '10_action_refinement_sr_pct': 100 * refinement_values.mean(),
        'refinement_minus_baseline_pp': 100 * (refinement_values.mean() - baseline_values.mean()),
        'delta_ci_low_pp': 100 * lo,
        'delta_ci_high_pp': 100 * hi,
        'failure_to_success': f_to_s,
        'success_to_failure': s_to_f,
        'paired_p_value': discordant_test(f_to_s, s_to_f),
    })

overall = summarize(paired).to_frame().T
by_suite = pd.DataFrame([
    {'suite': suite, **summarize(group).to_dict()}
    for suite, group in paired.groupby('suite', sort=True)])
print('Every exact-matched episode is in the SR denominator:')
display(overall)
display(by_suite[['suite', 'episodes', '10_action_baseline_sr_pct',
                  '10_action_refinement_sr_pct', 'refinement_minus_baseline_pp',
                  'failure_to_success', 'success_to_failure']])
paired.to_csv(OUTPUT / 'matched_episodes.csv', index=False)
overall.to_csv(OUTPUT / 'overall.csv', index=False)
by_suite.to_csv(OUTPUT / 'by_suite.csv', index=False)

## 4. Matched success rates and per-suite change

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(by_suite)); width = .38
labels = by_suite.suite.str.removeprefix('libero_')
axes[0].bar(x - width/2, by_suite['10_action_baseline_sr_pct'], width,
            label='10 actions, no refinement', color='#4C78A8')
axes[0].bar(x + width/2, by_suite['10_action_refinement_sr_pct'], width,
            label='10 actions, refinement', color='#F58518')
axes[0].set_xticks(x, labels, rotation=40, ha='right')
axes[0].set(ylabel='Success rate (%)', ylim=(0, 105),
            title='10-action refinement vs exact-matched baseline')
axes[0].legend(); axes[0].grid(axis='y', alpha=.2)

colors = np.where(by_suite.refinement_minus_baseline_pp >= 0, '#54A24B', '#E45756')
axes[1].bar(x, by_suite.refinement_minus_baseline_pp, color=colors)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].axhline(overall.refinement_minus_baseline_pp.iloc[0], color='#9467BD',
                linestyle='--',
                label=f"overall: {overall.refinement_minus_baseline_pp.iloc[0]:+.2f} pp")
axes[1].set_xticks(x, labels, rotation=40, ha='right')
axes[1].set(ylabel='Refinement minus baseline SR (percentage points)',
            title='Whole-matched-cohort SR change by suite')
axes[1].legend(); axes[1].grid(axis='y', alpha=.2)
fig.tight_layout()
fig.savefig(OUTPUT / 'source_action_horizon_10.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. Concise result

In [ ]:
result = overall.iloc[0]
print(f"Exact-matched episodes: {int(result.episodes)}")
print(f"10-action baseline:      {result['10_action_baseline_sr_pct']:.2f}%")
print(f"10-action refinement:    {result['10_action_refinement_sr_pct']:.2f}%")
print(f"Paired change:           {result.refinement_minus_baseline_pp:+.2f} pp "
      f"(95% CI {result.delta_ci_low_pp:+.2f} to {result.delta_ci_high_pp:+.2f})")
print(f"Transitions:             {int(result.failure_to_success)} F->S, "
      f"{int(result.success_to_failure)} S->F")